# Intro to NN & PyTorch — Part 1: Foundations (Lessons 1–5)

> 📚 **Part of a 3-notebook set** (Codecademy *Intro to NN with PyTorch*): **Part 1 — Foundations** · **Part 2 — Building Networks** · **Part 3 — Training**. Each notebook is self-contained (run its Setup cell first).

Tensors → Linear Regression → Perceptrons → Activation Functions → Multi-Layer Networks.

**Cell types:** 📖 markdown = concepts · 💻 code = exercises with real outputs · ❓ Q&A folded inline as block quotes.

In [1]:
# Setup — run once per session
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


torch version: 2.12.1+cu130
CUDA available: False


/home/plewis/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## Lesson 1 — Intro to Tensors

**Tensors** = the fundamental building blocks of neural networks in PyTorch. Like NumPy arrays, they're storage containers for numerical data.

> *Docs:* a tensor is a multi-dimensional array of values of the **same type**.

- Data usually starts as a **NumPy array** or **pandas DataFrame** → must be converted to tensors to use in PyTorch.
- Conversion uses **`torch.tensor()`**, which takes two args:
  1. the numerical data (NumPy array, Python list, or a numeric variable)
  2. the desired data type via the **`dtype`** parameter
- Common dtypes: **`torch.int`** (integers), **`torch.float`** (floats).

**Running example for this lesson:** build a NN to **predict apartment rent**, using the StreetEasy dataset from the original Intro to ML course.

### Converting a DataFrame → tensor
- Use the DataFrame's **`.values`** attribute to get a NumPy array first: `torch.tensor(df.values, dtype=torch.float)`.
- ⚠️ Selecting a **single column** can break due to torch's dimension assumptions. Two fixes:
  - **Double brackets** → keep it a full (2D) DataFrame: `df[['column']].values`
  - **`.view(-1, 1)`** → let torch auto-reshape into a column: `torch.tensor(df['column'].values, dtype=torch.float).view(-1, 1)`


In [2]:
# Ex.) Converting data to tensors
import numpy as np

# --- single value -> integer tensor ---
rent = 2550
rent_tensor = torch.tensor(rent, dtype=torch.int)
print("rent_tensor:", rent_tensor)

# --- numpy array -> float tensor (rent, size_sqft, age) ---
apt_array = np.array([2550, 750, 3.5])
apt_tensor = torch.tensor(apt_array, dtype=torch.float)
print("apt_tensor:", apt_tensor)

# --- DataFrame -> tensor (patterns; needs a real df to run) ---
# torch.tensor(df.values, dtype=torch.float)                       # whole frame
# torch.tensor(df[['column']].values, dtype=torch.float)           # one col, double brackets
# torch.tensor(df['column'].values, dtype=torch.float).view(-1, 1) # one col, reshape


rent_tensor: tensor(2550, dtype=torch.int32)
apt_tensor: tensor([2550.0000,  750.0000,    3.5000])


### Exercise — Tensors checkpoint (1–3)

1. Import `pandas`, `torch`, `numpy` (given).
2. Make an **int** tensor of `[2000, 500, 7]` → `apartment_tensor`.
3. Load `streeteasy.csv`, keep `rent`, `size_sqft`, `building_age_yrs`, and convert that DataFrame to a **`float32`** tensor → `apartments_tensor`.

> *Note: I don't have the real `streeteasy.csv`, so the cell builds a tiny fake DataFrame with the same 3 columns just so it runs. The conversion code (`torch.tensor(df.values, dtype=torch.float32)`) is exactly what the checkpoint asks for.*


In [3]:
# Checkpoint 1 — given imports
import pandas as pd
import torch
import numpy as np

# Checkpoint 2 — int tensor [2000, 500, 7]
apartment_array = np.array([2000, 500, 7])
apartment_tensor = torch.tensor(apartment_array, dtype=torch.int)
print("apartment_tensor:", apartment_tensor)

# Checkpoint 3 — DataFrame -> float32 tensor
# Real course code:  apartments_df = pd.read_csv("streeteasy.csv")
# FAKE stand-in data (same 3 columns) so this runs without the CSV:
apartments_df = pd.DataFrame({
    "rent":             [2550, 1800, 3200, 1450, 2750],
    "size_sqft":        [750,  600,  1100, 480,  900],
    "building_age_yrs": [3.5,  45,   12,   80,   7],
})

apartments_tensor = torch.tensor(apartments_df.values, dtype=torch.float32)

# show output
apartments_tensor


apartment_tensor: tensor([2000,  500,    7], dtype=torch.int32)


tensor([[2550.0000,  750.0000,    3.5000],
        [1800.0000,  600.0000,   45.0000],
        [3200.0000, 1100.0000,   12.0000],
        [1450.0000,  480.0000,   80.0000],
        [2750.0000,  900.0000,    7.0000]])

## Lesson 2 — Linear Regression Review

**Goal of this course's running project:** a NN to predict apartment **rent** from features like square footage.

- This is a **regression** problem → predicting a target **numeric value**.
- **Input features** → used to predict the **target / output** (in classification the target is often called a *label*).
  - e.g. input feature = square footage; target = predicted rent.

### Linear regression
The most basic regression. Uses a **linear equation** to predict. Standard line:

$$y = mx + b$$

Rent example with one feature:

$$\text{rent} = 2.5 \cdot \text{sqft} + 1000$$

- For a 500 sqft apartment: $2.5 \times 500 + 1000 = 2250$.

**ML / neural-network vocabulary** (vs. plain algebra):

| Algebra term | NN term | In the example |
|---|---|---|
| output ($y$) | **output / target** | rent |
| variable ($x$) | **input feature** | sqft |
| slope ($m$) | **weight** | 2.5 |
| intercept ($b$) | **bias** | 1000 |

### Multiple input features
Add each new feature **with its own weight**. Adding building age:

$$\text{rent} = 2.5 \cdot \text{sqft} - 1.5 \cdot \text{age} + 1000$$

- Weights can be **negative** (age has weight −1.5: older → cheaper).

**General form of any linear model:**
- some number of **input features**
- each feature × its own **weight** → a *weighted input feature*
- sum all weighted features **+ the bias**

> Even the most advanced NNs build on these same ideas — **inputs, weights, biases** — they just go beyond pure linear regression.


### Exercise — Linear regression checkpoint (1–3)

1. Model $\text{rent} = 3 \cdot \text{sz\_sqft} + 500$. Predict rent for a **500 sqft** apartment → `predicted_rent`.
2. Model $\text{rent} = 3 \cdot \text{sz\_sqft} + 10 \cdot \text{bedrooms} + 250$. **Weight** on `bedrooms`? → `bedroom_weight`.
3. Same model. **Bias**? → `bias`.

> Reading off a linear equation: each feature's coefficient is its **weight**; the lone constant term is the **bias**.


In [4]:
# Checkpoint 1 — predict rent for 500 sqft:  rent = 3*sz_sqft + 500
predicted_rent = 3 * 500 + 500
print(f"predicted rent = {predicted_rent}")

# Checkpoint 2 — rent = 3*sz_sqft + 10*bedrooms + 250 ; weight on bedrooms
bedroom_weight = 10

# Checkpoint 3 — bias (lone constant term)
bias = 250

print(f"bedroom_weight = {bedroom_weight}")
print(f"bias = {bias}")


predicted rent = 2000
bedroom_weight = 10
bias = 250


## Lesson 3 — Linear Regression with Perceptrons

First step toward NNs: turn the linear equation

$$\text{rent} = 2.5 \cdot \text{sqft} - 1.5 \cdot \text{age} + 1000$$

into a network structure called a **Perceptron**.

### What a Perceptron is
- A network of **nodes** (circles) connected by **edges** (arrows), arranged in vertical **layers**, flowing left → right.
- Defining trait: **one set of input nodes → a single output node.**
- The bias is drawn as an extra input node fixed at **1** with weight **1000**.

![Perceptron — weights on the edges](images/perceptron_1_weights.png)

### How it computes (forward pass)
Predict rent for a **10-year-old, 500 sqft** apartment:

**1. Feed inputs** into the input nodes (500 → sqft, 10 → age):

![Inputs fed in](images/perceptron_2_inputs.png)

**2. Multiply by edge weights** as values flow forward:

![Each input × its weight](images/perceptron_3_weighted.png)

**3. Sum** the weighted values at the output node (the `+`):

![Weighted values summed at output](images/perceptron_4_summed.png)

Result:

$$\text{rent} = 2.5 \times 500 - 1.5 \times 10 + 1000 \times 1$$

— **exactly the original linear equation.** The perceptron *is* the linear model, drawn as a graph.

> ⚠️ **Heads up:** eventually we stop treating the bias as its own input node (`1` × weight). The "1 node" trick just shows how the network reproduces a linear equation. **In PyTorch the bias is simply added onto the sum of the real inputs.**


> **Q:** *What is a perceptron?*
> **A:** The simplest neural-network structure: **one layer of input nodes → a single output node**. It's just a linear equation drawn as a graph.
> - Each input feature sits in an input node; each edge multiplies its input by a **weight**.
> - The output node **sums** the weighted inputs (**+ a bias**) → one number.
> - Optionally an **activation function** (e.g. ReLU) is applied to that sum for non-linearity.
>
> So $\text{rent} = 2.5\cdot\text{sqft} - 1.5\cdot\text{age} + 1000$ *is* a perceptron — weights = the coefficients, bias = the constant. Defining trait: **many inputs → one output node**. Stack many such nodes into hidden layers → the multi-layer networks of Lesson 5.


### Example — Perceptron computation in Python

The narrative network (sqft=2.5, age=−1.5, bias=1000), computing rent for a 500 sqft, 10 yr old apartment:

![Perceptron — weights on the edges](images/perceptron_1_weights.png)


In [5]:
# Define the inputs
size_sqft = 500.0
age = 10.0
bias = 1

# The inputs flow through the edges, receiving weights
weighted_size = 2.5 * size_sqft
weighted_age = -1.5 * age
weighted_bias = 1000 * bias

# The output node adds the weighted inputs
weighted_sum = weighted_size + weighted_age + weighted_bias

# Generate prediction
print("Predicted Rent:", weighted_sum)


Predicted Rent: 2235.0


### Exercise — add a `bedrooms` feature

New 4-input network: weights sqft=**3**, age=**−2.3**, bedrooms=**100**, bias=**500**.

![Perceptron with bedrooms input](images/perceptron_5_bedrooms.png)

Predict rent for: `size_sqft=1250.0`, `age=15.0`, `bedrooms=2.0` → save to `weighted_sum`.

> Adding a feature = add one input node + its own weighted term to the sum. Nothing else changes.


In [6]:
# Define the inputs
size_sqft = 1250.0
age = 15.0
bedrooms = 2.0
bias = 1.0

# The inputs flow through the edges, receiving weights
weighted_size = 3.0 * size_sqft
weighted_age = -2.3 * age
weighted_bedrooms = 100.0 * bedrooms
weighted_bias = 500.0 * bias

# The output node adds the weighted inputs
weighted_sum = weighted_size + weighted_age + weighted_bedrooms + weighted_bias

# Generate prediction
print("Predicted Rent:", weighted_sum)


Predicted Rent: 4415.5


## Lesson 4 — Activation Functions

So far the perceptron's output is just a **linear equation** → without more, NNs would only be fancy linear regression. What lifts them beyond it: **non-linear activation functions**, which let a network model **nonlinear relationships** (very common, impossible for linear regression).

![Perceptron producing the linear sum](images/perceptron_4_summed.png)

**Output node, two steps → now three:**
1. receive the weighted inputs
2. add them up *(this is the same linear equation as before)*
3. **apply an activation function** → introduces non-linearity

### ReLU
One of the most common activations. **Rectified Linear Unit:**
- negative input → returns **0**
- positive input → returns the number **unchanged**

$$\text{ReLU}(x) = \max(0, x)$$

- `ReLU(-1) = 0`, `ReLU(.5) = .5`
- Example with weighted inputs **3** and **−4**: add → `3 + (−4) = −1`; apply → `ReLU(−1) = 0`. **Output: 0.**

**Why zero-out negatives?**
- Lets **different sets of nodes be "active" (nonzero) for different inputs** — loosely brain-like (different neurons fire in different situations; though NNs don't really work like brains).
- Also **helps training via gradient descent** (covered later).

### Other activation functions
- **Sigmoid** — outputs only values **between 0 and 1**; common in **classification**.
- Many others exist; learn the standard one per application as it comes up.

**Diagram icons** (shaped like the function's plot):

| Icon shape | Meaning |
|---|---|
| horizontal line → diagonal | **ReLU** |
| sideways **S** | **Sigmoid** |
| **+** (plus sign) | no activation, just the weighted sum |


In [7]:
# (my own illustration, not a course exercise) ReLU & Sigmoid in PyTorch
import torch

x = torch.tensor([-4.0, -1.0, 0.0, 0.5, 3.0])

print("input:  ", x)
print("ReLU:   ", torch.relu(x))         # negatives -> 0, positives unchanged
print("Sigmoid:", torch.sigmoid(x))      # squashed into (0, 1)

# Lesson example: weighted inputs 3 and -4 into a ReLU node
print("\nReLU(3 + -4) =", torch.relu(torch.tensor(3.0 + -4.0)).item())


input:   tensor([-4.0000, -1.0000,  0.0000,  0.5000,  3.0000])
ReLU:    tensor([0.0000, 0.0000, 0.0000, 0.5000, 3.0000])
Sigmoid: tensor([0.0180, 0.2689, 0.5000, 0.6225, 0.9526])

ReLU(3 + -4) = 0.0


### Exercise — ReLU checkpoint (1–3)

1. Compute `ReLU(-3)`, `ReLU(0)`, `ReLU(3)` → `answer_1/2/3`.
2. Weighted inputs `-3.5` and `3`: compute the node output **with no activation** (`Linear_output`) and **with ReLU** (`ReLU_output`).
3. Weighted inputs `-2`, `1`, `.5`: fix the buggy `ReLU_output` so it returns the correct **0**.


In [8]:
# Checkpoint 1 — ReLU(-3), ReLU(0), ReLU(3)
answer_1 = 0.0
answer_2 = 0.0
answer_3 = 3.0

print('ReLU(-3) = ' + str(answer_1))
print('ReLU(0) = ' + str(answer_2))
print('ReLU(3) = ' + str(answer_3))


ReLU(-3) = 0.0
ReLU(0) = 0.0
ReLU(3) = 3.0


In [9]:
# Checkpoint 2 — weighted inputs -3.5 and 3
def ReLU(x):
    return max(0, x)

Linear_output = -3.5 + 3.0       # no activation, just the sum
ReLU_output = ReLU(-3.5 + 3.0)   # = ReLU(-0.5) = 0

print('ReLU node output: ' + str(ReLU_output))
print('Linear node output: ' + str(Linear_output))


ReLU node output: 0
Linear node output: -0.5


In [10]:
# Checkpoint 3 — fix the buggy ReLU output for inputs -2, 1, .5 (should be 0)
def ReLU(x):
    return max(0, x)

# FIXED: sum the weighted inputs first, then apply ReLU -> ReLU(-0.5) = 0
ReLU_output = ReLU(-2.0 + 1.0 + 0.5)

ReLU_output


0

## Lesson 5 — Multi-Layer Networks

So far: just **one input layer → one output layer**. Activation functions add nonlinearity, but to model **arbitrarily complex** datasets we need a deeper **multi-layer** structure — and we need to **train** it so it can learn the relationships.

### Hidden layers
**Hidden layers** = layers of nodes between the input and output layers. Data flows input → hidden(s) → output.

![Network with two hidden ReLU layers](images/multilayer_network.png)

Each hidden-layer node behaves like a perceptron. Precisely, every hidden node:
1. receives **weighted inputs** from all nodes in the **prior layer**
2. adds them up **with a bias term**
3. (optionally) applies an **activation function** to that weighted sum
4. sends the result to **every node in the next layer**

- Generally **all nodes in a given hidden layer use the same activation function** (e.g. "Hidden ReLU Layer").
- 🔧 **Convention change:** from now on, **no separate bias node** — just remember **every weighted sum includes a bias term**.

### The training process (~4 steps)
1. **Forward pass / feedforward** — push input through the network layer by layer to get the final output.
2. **Loss** — measure how close/far predictions are from the actual values.
3. **Backward pass / backpropagation** — use an optimization algorithm to go back and update the **weights and biases** to improve performance.
4. **Iterate** — repeat, checking each time whether the **loss (error) is going down**.

> Maps onto the 3B1B series: feedforward (Ch.1) → loss + gradient descent (Ch.2) → backprop (Ch.3–4). See [[bb-nn-video-notes]].
